"""
This script is used to plot the Global mean surface air temperature (GMSAT) from observation and multimodel simulation.
"""

## import observation data

In [ ]:
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.pyplot as plt
import matplotlib
import pandas as pd
from pathlib import Path
import glob
# %%
# define function
import src.SAT_function_Obs_Fingerprint as data_process
import src.Data_Preprocess as preprocess

In [ ]:
# subplot a: global mean temperature anomalies during 1850-2022 HadCRUT5
input_observation = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIG1/'

tas_HadCRUT_annual = xr.open_dataset(input_observation + 'tas_HadCRUT5_annual_anomalies.nc')

In [ ]:
tas_HadCRUT_annual

In [ ]:
tas_HadCRUT_annual_1850_2022 = tas_HadCRUT_annual.sel(year=slice('1993', '2022')).tas

In [ ]:
tas_HadCRUT5_annual_ano = tas_HadCRUT_annual_1850_2022.mean(dim=['year'])

In [ ]:
tas_HadCRUT5_annual_ano.min().values, tas_HadCRUT5_annual_ano.max().values

In [ ]:
# load the data
# MMEM GMSAT annual regression coefficient
dir_path = '/work/mh0033/m301036/OBS_LPS_revision/docs/data/FIGS5_S6/cesm2_100/'
HadCRUT_MMLE = xr.open_dataset(dir_path + 'OBS_SAT_anomaly_partition_wrt_MMLE_ENS.nc')
HadCRUT_MMLE

In [ ]:
beta_MMEM_HadCRUT5_annual = HadCRUT_MMLE['slope']
alpha_MMEM_HadCRUT5_annual = HadCRUT_MMLE['intercept']

In [ ]:
beta_MMEM_HadCRUT5_annual.min().values, beta_MMEM_HadCRUT5_annual.max().values

In [ ]:
alpha_MMEM_HadCRUT5_annual.min().values, alpha_MMEM_HadCRUT5_annual.max().values

### Subplot d: the reconstructed y mean and the y residual from the raw data

In [ ]:
GSAT_estimate_forced = HadCRUT_MMLE['forced_signal']
GSAT_estimate_internal = HadCRUT_MMLE['internal_variability']

In [ ]:
GSAT_estimate_forced, GSAT_estimate_internal 

In [ ]:
# Calculate the last 30years mean
GSAT_estimate_forced_1993_2022 = GSAT_estimate_forced.sel(year=slice('1993', '2022'))

In [ ]:
GSAT_estimate_forced_mean = GSAT_estimate_forced_1993_2022.mean(dim=['year'])
# GSAT_estimate_internal_mean = GSAT_estimate_internal['GSAT_HadCRUT5_Internal'].mean(dim=['year'])

In [ ]:
GSAT_estimate_forced_mean.min().values, GSAT_estimate_forced_mean.max().values

In [ ]:
GSAT_estimate_internal_1993_2022 = GSAT_estimate_internal.sel(year=slice('1993', '2022'))
GSAT_estimate_internal_mean = GSAT_estimate_internal_1993_2022.mean(dim=['year'])
GSAT_estimate_internal_mean.min().values, GSAT_estimate_internal_mean.max().values

## Plotting 

In [ ]:
import svgwrite
from matplotlib.lines import Line2D
from matplotlib.legend_handler import HandlerLine2D
from matplotlib.legend import Legend
import matplotlib.lines as Line2D

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as mticker
import cartopy.feature as cfeature
import cartopy.mpl.ticker as cticker
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import matplotlib.gridspec as gridspec
import matplotlib as mpl
import seaborn as sns
from matplotlib.colors import ListedColormap
from matplotlib.colors import BoundaryNorm, ListedColormap

In [ ]:
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.colors import BoundaryNorm
import cartopy.util as cutil
import seaborn as sns
import matplotlib.colors as mcolors
import palettable
#  cmap = mcolors.ListedColormap(palettable.scientific.diverging.Vik_20.mpl_colors)
cmap=mcolors.ListedColormap(palettable.cmocean.diverging.Balance_20.mpl_colors)

In [ ]:
from src.plot_func import * 

In [ ]:
# import matplotlib as mpl

# mpl.rcParams.update({
#     "figure.figsize": (7.25, 4.0),   # 2-column width, shallow height
#     "figure.dpi": 300,
#     "savefig.dpi": 600,

#     "font.family": "sans-serif",
#     "font.sans-serif": ["Myriad Pro", "Myriad", "Arial", "Helvetica", "DejaVu Sans"],
#     "font.size": 8,         # default text size
#     "axes.labelsize": 8,
#     "xtick.labelsize": 7,
#     "ytick.labelsize": 7,
#     "legend.fontsize": 7,

#     "axes.linewidth": 0.6,
#     "lines.linewidth": 0.8,
#     "xtick.major.width": 0.6,
#     "ytick.major.width": 0.6,
#     "xtick.minor.visible": True,
#     "ytick.minor.visible": True,
#     "xtick.direction": "out",
#     "ytick.direction": "out",
#     "xtick.major.size": 1.2,
#     "ytick.major.size": 1.2,

#     "axes.spines.top": True,
#     "axes.spines.right": True,
# })


In [ ]:
# Create the figure and a GridSpec layout
set_science_advances_style(column='double', height=4.0)

# 2) figure + gridspec
fig = plt.figure()   # use style's figsize
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.015, wspace=0.05)

# ===== Row 1: Two panels centered =====
ax0 = fig.add_subplot(gs[0, 0], projection=ccrs.Robinson(180))  # left (beta)
ax1 = fig.add_subplot(gs[0, 1], projection=ccrs.Robinson(180))  # right (alpha)

# ===== Row 2: Three panels full width =====
ax2 = fig.add_subplot(gs[1, 0], projection=ccrs.Robinson(180))  # y
ax3 = fig.add_subplot(gs[1, 1], projection=ccrs.Robinson(180))  # y_hat
ax4 = fig.add_subplot(gs[1, 2], projection=ccrs.Robinson(180))  # residuals

# === Plot 1: Beta and Alpha ===
bounds = np.arange(-1.0, 1.1, 0.1)
bounds_alpha = np.arange(-0.5, 0.55, 0.05)

beta_anomalies = beta_MMEM_HadCRUT5_annual
beta_anomalies_with_cyclic, lon_cyclic = cutil.add_cyclic_point(beta_anomalies, coord=beta_MMEM_HadCRUT5_annual['lon'])
contour_beta = plot_data(beta_anomalies_with_cyclic, beta_MMEM_HadCRUT5_annual['lat'], lon_cyclic, levels=bounds,
                         extend='both', cmap='RdBu_r', title="Regression Coefficient ("+r"$\beta$"+")", ax=ax0,
                         show_xticks=True, show_yticks=True)

alpha_anomalies = alpha_MMEM_HadCRUT5_annual
alpha_anomalies_with_cyclic, lon_cyclic = cutil.add_cyclic_point(alpha_anomalies, coord=alpha_MMEM_HadCRUT5_annual['lon'])
contour_alpha = plot_data(alpha_anomalies_with_cyclic, alpha_MMEM_HadCRUT5_annual['lat'], lon_cyclic, levels=bounds_alpha,
                          extend='both', cmap='RdBu_r', title="Intercept ("+r"$\alpha$"+")", ax=ax1,
                          show_xticks=True, show_yticks=False)

# === Plot 2: y, y_hat, residuals ===
level_forced = np.arange(-1.0, 1.1, 0.1)
level_internal = np.arange(-0.5, 0.55, 0.05)

tas_obs_with_cyclic, lon_cyclic = cutil.add_cyclic_point(tas_HadCRUT5_annual_ano, coord=tas_HadCRUT5_annual_ano['lon'])
contour_y = plot_data(tas_obs_with_cyclic, tas_HadCRUT5_annual_ano['lat'], lon_cyclic, levels=level_forced,
                      extend='both', cmap='RdBu_r', title=r'$y$', ax=ax2,
                      show_xticks=True, show_yticks=True)

forced_with_cyclic, lon_cyclic = cutil.add_cyclic_point(GSAT_estimate_forced_mean, coord=GSAT_estimate_forced_mean['lon'])
contour_yhat = plot_data(forced_with_cyclic, GSAT_estimate_forced_mean['lat'], lon_cyclic, levels=level_forced,
                         extend='both', cmap='RdBu_r', title=r'$\hat{y} = \beta \langle \bar{x}_t \rangle + \alpha$', ax=ax3,
                         show_xticks=True, show_yticks=False)

internal_with_cyclic, lon_cyclic = cutil.add_cyclic_point(GSAT_estimate_internal_mean, coord=GSAT_estimate_internal_mean['lon'])
contour_resid = plot_data(internal_with_cyclic, GSAT_estimate_internal_mean['lat'], lon_cyclic, levels=level_internal,
                          extend='both', cmap='RdBu_r', title=r'$\epsilon = y - \hat{y}$', ax=ax4,
                          show_xticks=True, show_yticks=False)
# Reduce tick label padding (bring ticks closer to plot)
for ax in [ax0, ax1, ax2, ax3, ax4]:
    ax.tick_params(pad=0)  # Adjust value: smaller = closer to plot
# === Colorbars ===
cbar_ax1 = fig.add_axes([0.15, 0.52, 0.2, 0.015])  # For beta
cbar1 = plt.colorbar(contour_beta, cax=cbar_ax1, orientation='horizontal')
cbar1.ax.tick_params(labelsize=5, which='major', length=2)
cbar1.ax.tick_params(which='minor', length=0)

cbar_ax2 = fig.add_axes([0.415, 0.52, 0.2, 0.015])  # For alpha
cbar2 = plt.colorbar(contour_alpha, cax=cbar_ax2, orientation='horizontal')
cbar2.ax.tick_params(labelsize=5, which='major', length=2)
cbar2.ax.tick_params(which='minor', length=0)

cbar_ax3 = fig.add_axes([0.2, 0.12, 0.35, 0.015])  # For y/yhat
cbar3 = plt.colorbar(contour_y, cax=cbar_ax3, orientation='horizontal')
cbar3.set_label('SAT anomalies (°C)', fontsize=5, labelpad=3)
cbar3.ax.tick_params(labelsize=5, which='major', length=2)
cbar3.ax.tick_params(which='minor', length=0)

cbar_ax4 = fig.add_axes([0.68, 0.12, 0.2, 0.015])  # For residuals
cbar4 = plt.colorbar(contour_resid, cax=cbar_ax4, orientation='horizontal')
cbar4.set_label('SAT anomalies (°C)', fontsize=5, labelpad=3)
cbar4.ax.tick_params(labelsize=5, which='major', length=2)
cbar4.ax.tick_params(which='minor', length=0)

# === Subplot labels ===
for ax, lab in zip([ax0, ax1, ax2, ax3, ax4], ['A', 'B', 'C', 'D', 'E']):
    ax.text(-0.015, 1.03, lab, transform=ax.transAxes,
            fontsize=8, fontweight='bold', ha='right', va='bottom')

# === Save figure ===
figure_output = '/work/mh0033/m301036/OBS_LPS_revision/docs/Figs/FIGS2/'
for ext in ("png", "pdf"):
    fig.savefig(figure_output+f"FIGURE_S2.{ext}", dpi=300, bbox_inches='tight')
fig.tight_layout()
plt.show()